## Plot fraction of individuals with low neutralization titers by strain

In [2]:
# Import packages
import os
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gmean


# Ignore error message from Altair about large dataframes
_ = alt.data_transformers.disable_max_rows()

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
    # '#EDEDF4', #white
    '#A499B3', #rose quartz
    '#515A47', #ebony
               ]

In [1]:
# Define inputs
resultsdir = '../results'
os.makedirs(resultsdir, exist_ok = True)

# Define titers
titers = pd.read_csv(
    ''
)

SyntaxError: incomplete input (1975094760.py, line 7)

In [4]:
# Define virus order
viral_plot_order = pd.read_csv('../../../data/H3N2library_2023-2024_strain_order.csv')
virus_order = [v for v in viral_plot_order.strain]

# Define vaccine strains
vaccine_strains = []
with open('../data/vaccine_strains.csv') as f:
    for line in f:
        line = line.strip('\n')
        if 'strain' not in line:
            vaccine_strains.append(line)

# Define egg-passaged vaccine strains
egg_passaged_vaccine_strains = []
with open('../data/egg-passaged_vaccine_strains.csv') as f:
    for line in f:
        line = line.strip('\n')
        if 'strain' not in line:
            egg_passaged_vaccine_strains.append(line)

# Define separate list where Massachusetts/18/2022 is reclassified as a 2023-circulating strains
vaccine_strains_no_Massachusetts = [item for item in vaccine_strains if item != 'A/Massachusetts/18/2022']

pre_2020_strains = [
    'A/Massachusetts/18/2022', 'A/Thailand/8/2022',
    'A/Darwin/6/2021', 'A/Darwin/9/2021',
]

vaccine_strains_pre_2020 = [item for item in vaccine_strains if item not in pre_2020_strains]

## Plots

In [5]:
def plot_titers_vaccination_cohorts(data, sort_order, _range = [30, 16000], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('timepoint', sort=['prevax']).scale(range=color_palette[4:])
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160
    width = 1100
    height = 200

    # Add vaccine strain weights
    vacc_weights = {
    'condition': [
        {'test' : 'datum.label == "A/Massachusetts/18/2022"', 'value': 'bold'},
        {'test' : 'datum.label == "A/Thailand/8/2022"', 'value': 'bold'},
        {'test' : 'datum.label == "A/Darwin/6/2021"', 'value': 'bold'},
        {'test' : 'datum.label == "A/Darwin/9/2021"', 'value': 'bold'},
    ],
     'value': 'normal'} # The default value if no condition is met

    band = (alt.Chart(data, width=width, height=height, )
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right',
                                                   labelFontWeight = vacc_weights,
                                         ),             
                          sort = virus_order),
                    alt.Y('titer', 
                          scale =alt.Scale(type='log',domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title="neutralization titer")),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data, width=width)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y('median(titer)'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .facet(row = alt.Row('group:N',title=None, sort=sort_order),
                      config = alt.Config(legend = alt.LegendConfig(titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                                                    strokeColor='gray',padding=10,cornerRadius=10,
                                                    labelLimit = 1000 # Let legend labels be as long as they want
                                                     )))
               .properties(title=title)
               .configure_header(labels=False, # Removing labels for pretty versions of figure, comment out to see labels
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                 # labelAngle=270,
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

In [7]:
data = all_titers[all_titers['group'].isin(
    # group_sort_list
    ['PennVaccineCohort', 'AusVaccineCohort']
)]
data = (data[~data['virus'].isin(vaccine_strains_no_Massachusetts)]
       .replace({'timepoint': {'d0': 'prevax',
                               'd28': 'postvax'}
                }))

plot = plot_titers_vaccination_cohorts(data, sort_order = ['PennVaccineCohort', 'AusVaccineCohort'], 
                                       _range=[30, 4000], title = '2023-circulating strains')
# Save final plot
outfile = os.path.join(resultsdir, 'post_vaccination_2023_titers.pdf')
plot.save(outfile, dpi = 600)
plot

alt.FacetChart(...)